In [1]:
import pandas as pd, numpy as np, joblib
from sklearn.neighbors import NearestNeighbors

veri = pd.read_csv("../data/clean/countries_clean.csv")
ulkeler = pd.read_csv("../data/clean/ulkeler_1m.csv")
paket = joblib.load("../models/kume_modeli.joblib")
F = paket["ozellikler"]
zc = ["z_" + f for f in F]

s = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time <= 2050)].copy()
s[zc] = paket["olcekleyici"].transform(s[F])
s["cluster"] = pd.Series(paket["model"].predict(s[zc].values), index=s.index).map(paket["etiket"])
havuz = s[s.Time <= 2023]
print(s.shape, havuz.shape)

(16261, 72) (11914, 72)


In [2]:
H = havuz[zc].values
iso = havuz.ISO3_code.values
uzak, sira = NearestNeighbors(n_neighbors=200).fit(H).kneighbors(H)
en_yakin_baska = np.array([uzak[i][iso[sira[i]] != iso[i]][0] for i in range(len(H))])
ESIK_MEDYAN, ESIK_90, ESIK_95, ESIK_99 = np.quantile(en_yakin_baska, [0.5, 0.9, 0.95, 0.99])
print(round(ESIK_MEDYAN, 3), round(ESIK_90, 3), round(ESIK_95, 3), round(ESIK_99, 3))

0.044 0.11 0.149 0.257


In [3]:
def etiket(m):
    if m <= ESIK_MEDYAN:
        return "Çok benzer"
    return "Benzer" if m <= ESIK_95 else "Tarihte benzeri yok"

satir = []
for yil in [2023, 2050]:
    for r in s[s.Time == yil].itertuples():
        q = np.array([getattr(r, c) for c in zc])
        p = havuz[havuz.ISO3_code != r.ISO3_code]
        p = p.assign(mesafe=np.sqrt(((p[zc].values - q) ** 2).sum(axis=1)))
        p = p.loc[p.groupby("ISO3_code").mesafe.idxmin()].sort_values("mesafe")
        icinde = p[p.mesafe <= ESIK_95].head(3)
        secilen = icinde if len(icinde) else p.head(1)
        for i, t in enumerate(secilen.itertuples(), start=1):
            satir.append({"iso3": r.ISO3_code, "country": r.Location, "query_year": yil, "rank": i,
                          "twin_iso3": t.ISO3_code, "twin_country": t.Location, "twin_year": t.Time,
                          "distance": round(t.mesafe, 3), "label": etiket(t.mesafe)})
ikiz = pd.DataFrame(satir)

ilk = ikiz[ikiz["rank"] == 1]
print(pd.crosstab(ilk.query_year, ilk.label))
print(ikiz[ikiz.iso3 == "TUR"].to_string(index=False))

label       Benzer  Tarihte benzeri yok  Çok benzer
query_year                                         
2023            78                    7          76
2050            63                   47          51
iso3 country  query_year  rank twin_iso3               twin_country  twin_year  distance      label
 TUR Türkiye        2023     1       NLD                Netherlands       1987     0.043 Çok benzer
 TUR Türkiye        2023     2       IRN Iran (Islamic Republic of)       2022     0.052     Benzer
 TUR Türkiye        2023     3       CUB                       Cuba       2001     0.061     Benzer
 TUR Türkiye        2050     1       SVN                   Slovenia       2023     0.088     Benzer
 TUR Türkiye        2050     2       PRT                   Portugal       2019     0.109     Benzer
 TUR Türkiye        2050     3       JPN                      Japan       2011     0.127     Benzer


In [4]:
for ad, e in [("%90", ESIK_90), ("%95", ESIK_95), ("%99", ESIK_99)]:
    print(ad, {y: int((ilk[ilk.query_year == y].distance > e).sum()) for y in [2023, 2050]})

%90 {2023: 12, 2050: 57}
%95 {2023: 7, 2050: 47}
%99 {2023: 0, 2050: 22}


In [5]:
ov = pd.read_csv("../data/raw/WPP2024_Demographic_Indicators_OtherVariants_csv", encoding="utf-8-sig",
                 low_memory=False,
                 usecols=["ISO3_code", "Variant", "Time", "TFR", "LEx", "MedianAgePop", "TPopulation1July"])
ov = ov[ov.ISO3_code.isin(ulkeler.iso3) & ov.Variant.isin(["Lower 95 PI", "Upper 95 PI"])
        & ov.Time.between(2024, 2050)]
bant = ov.pivot_table(index=["ISO3_code", "Time"], columns="Variant",
                      values=["TFR", "LEx", "MedianAgePop", "TPopulation1July"])
bant.columns = [f"{a}_{'lo95' if 'Lower' in b else 'hi95'}" for a, b in bant.columns]
bant = bant.reset_index()
print(bant.shape)

(4347, 10)


In [6]:
x = s.merge(bant, on=["ISO3_code", "Time"], how="left")
seri = pd.DataFrame({
    "iso3": x.ISO3_code, "country": x.Location, "region": x.Region, "year": x.Time,
    "is_projection": x.Time > 2023,
    "cluster": x.cluster, "cluster_name": x.cluster.map(paket["isimler"]),
    "tfr": x.TFR, "tfr_lo95": x.TFR_lo95, "tfr_hi95": x.TFR_hi95,
    "lex": x.LEx, "lex_lo95": x.LEx_lo95, "lex_hi95": x.LEx_hi95,
    "median_age": x.MedianAgePop, "median_age_lo95": x.MedianAgePop_lo95,
    "median_age_hi95": x.MedianAgePop_hi95,
    "pop_millions": (x.TPopulation1July / 1000).round(3),
    "pop_lo95_millions": (x.TPopulation1July_lo95 / 1000).round(3),
    "pop_hi95_millions": (x.TPopulation1July_hi95 / 1000).round(3)})
seri = seri.sort_values(["iso3", "year"])
seri["back_step"] = seri.groupby("iso3").cluster.diff() < 0

# Kontroller
p = seri[seri.is_projection]
print("Projeksiyon yıllarında boş bant:", int(p[["tfr_lo95", "pop_hi95_millions"]].isna().sum().sum()))
print("Orta senaryo bandın içinde mi:",
      ((p.tfr >= p.tfr_lo95 - 1e-9) & (p.tfr <= p.tfr_hi95 + 1e-9)).mean())
print("Geri adım 1950–2023:", int(seri[~seri.is_projection].back_step.sum()),
      "| 2024–2050:", int(p.back_step.sum()))

seri.to_csv("../data/out/g10_pano_seriler.csv", index=False)
ikiz.to_csv("../data/out/g10_pano_ikizler.csv", index=False)
print("Kaydedildi:", seri.shape, ikiz.shape)

Projeksiyon yıllarında boş bant: 0
Orta senaryo bandın içinde mi: 1.0
Geri adım 1950–2023: 44 | 2024–2050: 0
Kaydedildi: (16261, 20) (806, 9)
